# Multimodal video models and paired errors

Recompute seed-42 text and Equal VAT classification scores, confusion matrices and paired decision transitions from stored out-of-fold predictions.

**Scope:** These notebooks recompute descriptive statistics from the archived data and predictions. They do not retrain the original encoders or produce new evaluation populations.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, confusion_matrix, roc_auc_score, average_precision_score, brier_score_loss

ROOT = Path.cwd().resolve()
if not (ROOT / "CITATION.cff").exists():
    ROOT = ROOT.parent
assert (ROOT / "CITATION.cff").exists(), "Run from the repository root or notebooks/"
print("Project:", ROOT.name)

## Full configuration summary
Selected threshold, fixed 0.5 and three-seed summary are distinct measurements.

In [ ]:
configs=pd.read_csv(ROOT / "results/statistics/ALL_CONFIGURATION_METRICS.csv")
three=pd.read_csv(ROOT / "results/video/gold_adjudicated_v1.1/THREE_SEED_SUMMARY.csv")
display(configs.sort_values("macro_f1",ascending=False).head(21))
display(three[three.model.isin(["text","equal_vat","visual","audio"])])

## Independent paired predictions

In [ ]:
v=pd.read_csv(ROOT / "results/video/gold_adjudicated_v1.1/VIDEO_OOF_PREDICTIONS.csv")
s=v[v.seed==42].copy()
assert len(s)==3951 and s.group_id.nunique()==214
y=s.label_final.to_numpy(); pred_text=s.text_prediction.to_numpy();pred_vat=s.equal_vat_prediction.to_numpy()
rows=[]
for label,yp in [("Extracted text",pred_text),("Equal VAT",pred_vat)]:
    cm=confusion_matrix(y,yp,labels=[0,1])
    rows.append({"model":label,"macro_f1":f1_score(y,yp,average="macro",labels=[0,1]),"TN":int(cm[0,0]),"FP":int(cm[0,1]),"FN":int(cm[1,0]),"TP":int(cm[1,1])})
display(pd.DataFrame(rows))
assert (int((pred_text!=y).sum()),int((pred_vat!=y).sum())) == (1639,1655)
corrected = int(((pred_text!=y)&(pred_vat==y)).sum())
introduced = int(((pred_text==y)&(pred_vat!=y)).sum())
print("Text errors corrected by VAT:",corrected,"New errors introduced:",introduced,"Net additional errors:",introduced-corrected)
assert (corrected,introduced)==(308,324)

## Two confusion matrices

In [ ]:
for label,pred in [("Extracted text",pred_text),("Equal VAT",pred_vat)]:
    mat=confusion_matrix(y,pred,labels=[0,1]); fig,ax=plt.subplots(figsize=(4.0,3.5))
    ax.imshow(mat,cmap="Greys",vmin=0,vmax=mat.max())
    for i in range(2):
        for j in range(2):ax.text(j,i,str(mat[i,j]),ha="center",va="center")
    ax.set(xticks=[0,1],yticks=[0,1],xticklabels=["Negative","Positive"],yticklabels=["Negative","Positive"],xlabel="Predicted",ylabel="Reference",title=label)
    plt.tight_layout();plt.show()